In [ ]:
print('Hello')

In [ ]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
from pathlib import Path

DATASET = Path("/kaggle/input/datasets/smnahian/lowlight")

print("Dataset exists:", DATASET.exists())
print("Contents:")

for item in DATASET.iterdir():
    print(item)

In [ ]:
from pathlib import Path
import pandas as pd

DATASET = Path("/kaggle/input/datasets/smnahian/lowlight")

IMAGE_DIR = DATASET / "Images"
ANNOTATION_DIR = DATASET / "Annotaions"
SPLIT_FILE = ANNOTATION_DIR / "imageclasslist.txt"

print("Image directory exists:", IMAGE_DIR.exists())
print("Annotation directory exists:", ANNOTATION_DIR.exists())
print("Split file exists:", SPLIT_FILE.exists())

print("\nImage folders:")
for folder in sorted(IMAGE_DIR.iterdir()):
    print(folder.name)

print("\nAnnotation folders:")
for folder in sorted(ANNOTATION_DIR.iterdir()):
    print(folder.name)

In [ ]:
from pathlib import Path
import pandas as pd

DATASET = Path("/kaggle/input/datasets/smnahian/lowlight")
SPLIT_FILE = DATASET / "Annotaions" / "imageclasslist.txt"

# Read the file:
# Data rows are whitespace-separated.
df = pd.read_csv(
    SPLIT_FILE,
    sep=r"\s+",
    skiprows=1,
    header=None,
    names=["Name", "Class", "Light", "In_Out", "Split"]
)

# Convert numeric columns
df["Class"] = pd.to_numeric(df["Class"], errors="coerce")
df["Light"] = pd.to_numeric(df["Light"], errors="coerce")
df["In_Out"] = pd.to_numeric(df["In_Out"], errors="coerce")
df["Split"] = pd.to_numeric(df["Split"], errors="coerce")

# Remove any invalid rows
df = df.dropna(subset=["Name", "Class", "Split"])

# Convert to integers
df["Class"] = df["Class"].astype(int)
df["Light"] = df["Light"].astype(int)
df["In_Out"] = df["In_Out"].astype(int)
df["Split"] = df["Split"].astype(int)

print("Columns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
print(df.head())

print("\nTotal rows:", len(df))

In [ ]:
# ExDark class IDs
vehicle_classes = {
    1: "Bicycle",
    2: "Boat",
    4: "Bus",
    5: "Car",
    10: "Motorbike"
}

# Keep only vehicle classes
vehicle_df = df[df["Class"].isin(vehicle_classes.keys())].copy()

# Convert class IDs to names
vehicle_df["Class_Name"] = vehicle_df["Class"].map(vehicle_classes)

# Convert split IDs
split_names = {
    1: "Train",
    2: "Validation",
    3: "Test"
}

vehicle_df["Split_Name"] = vehicle_df["Split"].map(split_names)

print("Total vehicle images:", len(vehicle_df))

print("\nClass counts:")
print(vehicle_df["Class_Name"].value_counts().sort_index())

print("\nSplit counts:")
print(vehicle_df["Split_Name"].value_counts())

print("\nClass × Split:")
print(
    pd.crosstab(
        vehicle_df["Class_Name"],
        vehicle_df["Split_Name"]
    )
)

In [ ]:
from pathlib import Path

image_dir = DATASET / "Images" / "Boat"
annotation_dir = DATASET / "Annotaions" / "Boat"

# Get first image
images = list(image_dir.glob("*"))
image_file = images[0]

print("Image:")
print(image_file)

# Annotation filename = complete image filename + ".txt"
annotation_file = annotation_dir / (image_file.name + ".txt")

print("\nAnnotation:")
print(annotation_file)

print("\nAnnotation exists:", annotation_file.exists())

if annotation_file.exists():
    print("\nAnnotation content:\n")
    print(annotation_file.read_text(errors="ignore"))

In [ ]:
from pathlib import Path
from collections import Counter

# =========================
# Dataset paths
# =========================

DATASET = Path("/kaggle/input/datasets/smnahian/lowlight")

IMAGE_DIR = DATASET / "Images"
ANNOTATION_DIR = DATASET / "Annotaions"

# Only these 5 classes
VEHICLE_CLASSES = {
    "Bicycle",
    "Boat",
    "Bus",
    "Car",
    "Motorbike"
}

# =========================
# Counters
# =========================

total_images = 0
annotation_found = 0
annotation_missing = 0

images_with_vehicle = 0
images_with_non_vehicle = 0

vehicle_objects = Counter()
non_vehicle_objects = Counter()

missing_annotations = []
empty_annotations = []
invalid_annotations = []

# =========================
# Scan vehicle image folders
# =========================

for class_name in sorted(VEHICLE_CLASSES):

    image_folder = IMAGE_DIR / class_name
    annotation_folder = ANNOTATION_DIR / class_name

    print(f"Scanning {class_name}...")

    for image_file in image_folder.iterdir():

        if not image_file.is_file():
            continue

        total_images += 1

        # Annotation name:
        # 2015_01231.jpg -> 2015_01231.jpg.txt
        annotation_file = annotation_folder / (image_file.name + ".txt")

        if not annotation_file.exists():
            annotation_missing += 1
            missing_annotations.append(str(image_file))
            continue

        annotation_found += 1

        # Read annotation
        lines = annotation_file.read_text(
            errors="ignore"
        ).splitlines()

        objects_in_image = []
        has_vehicle = False
        has_non_vehicle = False

        for line in lines:

            line = line.strip()

            # Skip metadata line
            if not line or line.startswith("%"):
                continue

            parts = line.split()

            # Expected:
            # Class l t w h + 7 values
            if len(parts) < 5:
                invalid_annotations.append(
                    (str(annotation_file), line)
                )
                continue

            object_class = parts[0]

            objects_in_image.append(object_class)

            if object_class in VEHICLE_CLASSES:
                vehicle_objects[object_class] += 1
                has_vehicle = True
            else:
                non_vehicle_objects[object_class] += 1
                has_non_vehicle = True

        if not objects_in_image:
            empty_annotations.append(str(annotation_file))

        if has_vehicle:
            images_with_vehicle += 1

        if has_non_vehicle:
            images_with_non_vehicle += 1


# =========================
# Results
# =========================

print("\n" + "=" * 50)
print("DATASET VALIDATION RESULT")
print("=" * 50)

print(f"\nTotal vehicle-class images: {total_images}")

print(f"Annotations found:          {annotation_found}")
print(f"Annotations missing:        {annotation_missing}")

print(f"\nImages containing vehicle:  {images_with_vehicle}")
print(f"Images containing non-vehicle: {images_with_non_vehicle}")

print("\nVehicle object counts:")
for name, count in vehicle_objects.items():
    print(f"  {name}: {count}")

print("\nNon-vehicle object counts:")
for name, count in non_vehicle_objects.most_common():
    print(f"  {name}: {count}")

print("\nEmpty annotations:", len(empty_annotations))
print("Invalid annotations:", len(invalid_annotations))

# Show missing annotation examples
if missing_annotations:
    print("\nFirst 10 missing annotation examples:")
    for x in missing_annotations[:10]:
        print(" ", x)

# Show invalid annotation examples
if invalid_annotations:
    print("\nFirst 10 invalid annotation examples:")
    for x in invalid_annotations[:10]:
        print(" ", x)

In [ ]:
# Check images with missing/empty annotations

print("Missing annotation images:")
for x in missing_annotations:
    print(x)

print("\nEmpty annotation files:")
for x in empty_annotations:
    print(x)

In [ ]:
from pathlib import Path

# =========================
# Paths
# =========================

DATASET = Path("/kaggle/input/datasets/smnahian/lowlight")

IMAGE_DIR = DATASET / "Images"
ANNOTATION_DIR = DATASET / "Annotaions"

# Output directory
OUTPUT_DIR = Path("/kaggle/working/filtered_annotations")

# Create output directory
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# =========================
# Vehicle classes
# =========================

VEHICLE_CLASSES = {
    "Bicycle",
    "Boat",
    "Bus",
    "Car",
    "Motorbike"
}


# =========================
# Counters
# =========================

total_annotations = 0
filtered_annotations = 0
removed_objects = 0
images_with_vehicle = 0
empty_after_filtering = 0


# =========================
# Process annotations
# =========================

for class_folder in sorted(VEHICLE_CLASSES):

    input_folder = ANNOTATION_DIR / class_folder

    # Create corresponding output folder
    output_folder = OUTPUT_DIR / class_folder
    output_folder.mkdir(parents=True, exist_ok=True)

    print(f"Processing {class_folder}...")

    # Find all annotation files
    for annotation_file in input_folder.glob("*.txt"):

        # Read annotation
        lines = annotation_file.read_text(
            errors="ignore"
        ).splitlines()

        total_annotations += 1

        filtered_lines = []

        for line in lines:

            line = line.strip()

            # Keep metadata line
            if line.startswith("%"):
                filtered_lines.append(line)
                continue

            # Skip blank lines
            if not line:
                continue

            parts = line.split()

            # Need at least class + l + t + w + h
            if len(parts) < 5:
                continue

            object_class = parts[0]

            # Keep only vehicle objects
            if object_class in VEHICLE_CLASSES:

                filtered_lines.append(line)
                filtered_annotations += 1

            else:

                removed_objects += 1

        # Save filtered annotation
        output_file = output_folder / annotation_file.name

        if len(filtered_lines) > 1:
            images_with_vehicle += 1

        else:
            empty_after_filtering += 1

        output_file.write_text(
            "\n".join(filtered_lines)
        )


# =========================
# Results
# =========================

print("\n" + "=" * 50)
print("ANNOTATION FILTERING COMPLETE")
print("=" * 50)

print(f"Original annotation files processed: {total_annotations}")
print(f"Vehicle annotations kept:             {filtered_annotations}")
print(f"Non-vehicle annotations removed:      {removed_objects}")
print(f"Files containing vehicle objects:     {images_with_vehicle}")
print(f"Files empty after filtering:          {empty_after_filtering}")

print("\nFiltered annotations saved to:")
print(OUTPUT_DIR)

In [ ]:
# Check the same Boat annotation we inspected earlier

original_file = (
    ANNOTATION_DIR
    / "Boat"
    / "2015_00658.jpg.txt"
)

filtered_file = (
    OUTPUT_DIR
    / "Boat"
    / "2015_00658.jpg.txt"
)

print("Original annotation:\n")
print(original_file.read_text(errors="ignore"))

print("\n" + "=" * 50)

print("Filtered annotation:\n")
print(filtered_file.read_text(errors="ignore"))

In [ ]:
from pathlib import Path
import shutil

# =========================
# Paths
# =========================

DATASET = Path("/kaggle/input/datasets/smnahian/lowlight")

IMAGE_DIR = DATASET / "Images"
FILTERED_ANNOTATION_DIR = Path("/kaggle/working/filtered_annotations")

# New clean dataset
CLEAN_DATASET = Path("/kaggle/working/vehicle_dataset")

CLEAN_IMAGE_DIR = CLEAN_DATASET / "Images"
CLEAN_ANNOTATION_DIR = CLEAN_DATASET / "Annotations"

# Vehicle classes
VEHICLE_CLASSES = {
    "Bicycle",
    "Boat",
    "Bus",
    "Car",
    "Motorbike"
}

# Create output folders
for class_name in VEHICLE_CLASSES:
    (CLEAN_IMAGE_DIR / class_name).mkdir(parents=True, exist_ok=True)
    (CLEAN_ANNOTATION_DIR / class_name).mkdir(parents=True, exist_ok=True)


# =========================
# Copy valid image + annotation pairs
# =========================

copied_images = 0
copied_annotations = 0
skipped_images = 0

for class_name in sorted(VEHICLE_CLASSES):

    image_folder = IMAGE_DIR / class_name
    annotation_folder = FILTERED_ANNOTATION_DIR / class_name

    output_image_folder = CLEAN_IMAGE_DIR / class_name
    output_annotation_folder = CLEAN_ANNOTATION_DIR / class_name

    for image_file in image_folder.iterdir():

        if not image_file.is_file():
            continue

        # Filtered annotation uses:
        # image.jpg -> image.jpg.txt
        annotation_file = annotation_folder / (image_file.name + ".txt")

        # If annotation doesn't exist, skip
        if not annotation_file.exists():
            skipped_images += 1
            continue

        # Read filtered annotation
        annotation_text = annotation_file.read_text(errors="ignore").strip()

        # Remove metadata line and check whether
        # at least one actual vehicle object exists
        object_lines = [
            line.strip()
            for line in annotation_text.splitlines()
            if line.strip() and not line.strip().startswith("%")
        ]

        # If no vehicle object, skip
        if len(object_lines) == 0:
            skipped_images += 1
            continue

        # Copy image
        shutil.copy2(
            image_file,
            output_image_folder / image_file.name
        )

        # Copy filtered annotation
        shutil.copy2(
            annotation_file,
            output_annotation_folder / annotation_file.name
        )

        copied_images += 1
        copied_annotations += 1


# =========================
# Results
# =========================

print("=" * 55)
print("CLEAN VEHICLE DATASET CREATED")
print("=" * 55)

print(f"Images copied:       {copied_images}")
print(f"Annotations copied:  {copied_annotations}")
print(f"Images skipped:      {skipped_images}")

print("\nDataset location:")
print(CLEAN_DATASET)

In [ ]:
print("\nClass-wise image counts:")

for class_name in sorted(VEHICLE_CLASSES):
    folder = CLEAN_IMAGE_DIR / class_name
    count = len([x for x in folder.iterdir() if x.is_file()])
    print(f"{class_name}: {count}")

print("\nClass-wise annotation counts:")

for class_name in sorted(VEHICLE_CLASSES):
    folder = CLEAN_ANNOTATION_DIR / class_name
    count = len([x for x in folder.iterdir() if x.is_file()])
    print(f"{class_name}: {count}")

In [ ]:
!git clone https://github.com/caiyuanhao1998/Retinexformer.git
%cd Retinexformer
!ls

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
!cat setup.py

In [ ]:
!find Enhancement -maxdepth 2 -type f | head -30

In [ ]:
!sed -n '1,240p' Enhancement/test_from_dataset.py

In [ ]:
!sed -n '1,240p' Enhancement/utils.py

In [ ]:
!grep -n -i -A5 -B5 "pretrained" README.md | head -80

!find . -maxdepth 3 -type f | grep -E "\.(pth|pt|ckpt)$"

In [ ]:
!mkdir -p pretrained_weights

In [ ]:
!pip install -q gdown

In [ ]:
!ls -lh pretrained_weights/

In [ ]:
import requests

url = "https://drive.google.com/drive/folders/1ynK5hfQachzc8y96ZumhkPPDXzHJwaQV"

r = requests.get(url)

print("Status:", r.status_code)
print("Page length:", len(r.text))
print("LOL_v2_real.pth found:", "LOL_v2_real.pth" in r.text)

In [ ]:
import re
import requests

url = "https://drive.google.com/drive/folders/1ynK5hfQachzc8y96ZumhkPPDXzHJwaQV"

html = requests.get(url).text

# Find the file ID associated with LOL_v2_real.pth
pattern = r'([a-zA-Z0-9_-]{20,})[^"]{0,500}LOL_v2_real\.pth'
matches = re.findall(pattern, html)

print("Possible IDs found:")
for x in matches[:10]:
    print(x)

In [ ]:
!gdown "https://drive.google.com/uc?id=1xDwQtTCj3tlAVCTJgYrzonBGVwqeOhKu" \
    -O pretrained_weights/LOL_v2_real.pth

In [ ]:
import os

weight_path = "pretrained_weights/LOL_v2_real.pth"

print("Exists:", os.path.exists(weight_path))

if os.path.exists(weight_path):
    print("Size:", round(os.path.getsize(weight_path) / (1024**2), 2), "MB")

In [ ]:
!pip install -q lmdb
import sys
import torch

sys.path.insert(0, "/kaggle/working/Retinexformer")

from basicsr.models import create_model
from basicsr.utils.options import parse

print("RetinexFormer imports: OK")
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

In [ ]:
import yaml

opt_path = "Options/RetinexFormer_LOL_v2_real.yml"

with open(opt_path, "r") as f:
    config = yaml.safe_load(f)

print("Configuration loaded successfully")
print("Network:", config["network_g"].get("type"))

In [ ]:
from pathlib import Path

test_image = Path(
    "/kaggle/working/vehicle_dataset/Images/Boat/2015_01231.jpg"
)

print("Image exists:", test_image.exists())
print("Image:", test_image)

In [ ]:
import os
import cv2
import numpy as np
import torch
import torch.nn.functional as F

from basicsr.models import create_model
from basicsr.utils.options import parse

# ============================================================
# PATHS
# ============================================================

repo_dir = "/kaggle/working/Retinexformer"

opt_path = os.path.join(
    repo_dir,
    "Options/RetinexFormer_LOL_v2_real.yml"
)

weight_path = os.path.join(
    repo_dir,
    "pretrained_weights/LOL_v2_real.pth"
)

input_path = (
    "/kaggle/working/vehicle_dataset/Images/"
    "Boat/2015_01231.jpg"
)

output_path = (
    "/kaggle/working/test_retinexformer.jpg"
)


# ============================================================
# LOAD CONFIGURATION
# ============================================================

opt = parse(
    opt_path,
    is_train=False
)

opt["dist"] = False

model = create_model(opt).net_g


# ============================================================
# LOAD PRETRAINED WEIGHTS
# ============================================================

checkpoint = torch.load(
    weight_path,
    map_location="cpu"
)

try:

    model.load_state_dict(
        checkpoint["params"]
    )

except Exception:

    new_checkpoint = {}

    for k, v in checkpoint["params"].items():

        new_checkpoint["module." + k] = v

    model.load_state_dict(
        new_checkpoint
    )


# ============================================================
# MOVE MODEL TO GPU
# ============================================================

model = model.cuda()
model.eval()

print("=" * 60)
print("RETINEXFORMER MODEL LOADED")
print("=" * 60)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

print("Model loaded successfully.")


# ============================================================
# LOAD IMAGE
# ============================================================

img = cv2.imread(
    input_path
)

if img is None:

    raise FileNotFoundError(
        input_path
    )

print("\nOriginal image shape:", img.shape)

# BGR -> RGB
img = cv2.cvtColor(
    img,
    cv2.COLOR_BGR2RGB
)

# [0,255] -> [0,1]
img = img.astype(
    np.float32
) / 255.0

# H,W,C -> C,H,W
input_tensor = (
    torch
    .from_numpy(img)
    .permute(2, 0, 1)
)

# Add batch dimension
input_tensor = (
    input_tensor
    .unsqueeze(0)
    .cuda()
)


# ============================================================
# ORIGINAL DIMENSIONS
# ============================================================

_, _, h, w = input_tensor.shape

print(
    "Input tensor size:",
    f"{h} x {w}"
)


# ============================================================
# PADDING
# RetinexFormer requires dimensions divisible by 4
# ============================================================

factor = 4

new_h = (
    (h + factor - 1)
    // factor
) * factor

new_w = (
    (w + factor - 1)
    // factor
) * factor

pad_h = new_h - h
pad_w = new_w - w

if pad_h > 0 or pad_w > 0:

    input_tensor = F.pad(
        input_tensor,
        (0, pad_w, 0, pad_h),
        mode="reflect"
    )

print(
    "Padded tensor size:",
    f"{new_h} x {new_w}"
)


# ============================================================
# RETINEXFORMER ENHANCEMENT
# ============================================================

with torch.inference_mode():

    restored = model(
        input_tensor
    )


# ============================================================
# REMOVE PADDING
# ============================================================

restored = restored[
    :, :, :h, :w
]


# ============================================================
# CLAMP VALUES
# ============================================================

restored = torch.clamp(
    restored,
    0,
    1
)


# ============================================================
# TENSOR -> NUMPY
# ============================================================

restored = (
    restored
    .cpu()
    .squeeze(0)
    .permute(1, 2, 0)
    .numpy()
)


# ============================================================
# [0,1] -> [0,255]
# ============================================================

restored = (
    restored * 255
).astype(
    np.uint8
)


# ============================================================
# RGB -> BGR
# ============================================================

restored = cv2.cvtColor(
    restored,
    cv2.COLOR_RGB2BGR
)


# ============================================================
# SAVE IMAGE
# ============================================================

save_success = cv2.imwrite(
    output_path,
    restored
)

if not save_success:

    raise RuntimeError(
        "Failed to save enhanced image."
    )


# ============================================================
# RESULT
# ============================================================

print("\n" + "=" * 60)
print("RETINEXFORMER TEST COMPLETE")
print("=" * 60)

print("Enhancement successful.")
print("Saved to:")
print(output_path)

print(
    "Enhanced image shape:",
    restored.shape
)

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

original = Image.open(
    "/kaggle/working/vehicle_dataset/Images/Boat/2015_01231.jpg"
)

enhanced = Image.open(
    "/kaggle/working/test_retinexformer.jpg"
)

print("Original size:", original.size)
print("Enhanced size:", enhanced.size)

plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
plt.imshow(original)
plt.title("Original ExDark")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(enhanced)
plt.title("RetinexFormer Enhanced")
plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# শুধু একটি image enhance করার function

def enhance_one(input_path, output_path):

    img = cv2.imread(input_path)

    if img is None:
        print("Image not found:", input_path)
        return

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = np.float32(img) / 255.0

    tensor = torch.from_numpy(img).permute(2, 0, 1)
    tensor = tensor.unsqueeze(0).cuda()

    _, _, h, w = tensor.shape

    # Padding
    factor = 4
    new_h = ((h + factor - 1) // factor) * factor
    new_w = ((w + factor - 1) // factor) * factor

    tensor = F.pad(
        tensor,
        (0, new_w - w, 0, new_h - h),
        mode="reflect"
    )

    # RetinexFormer
    with torch.inference_mode():
        restored = model(tensor)

    # Remove padding
    restored = restored[:, :, :h, :w]

    # Convert back
    restored = torch.clamp(restored, 0, 1)
    restored = (
        restored.cpu()
        .permute(0, 2, 3, 1)
        .squeeze(0)
        .numpy()
    )

    restored = (restored * 255).astype(np.uint8)

    # Save
    Image.fromarray(restored).save(output_path)

    print("Done!")
    print("Original:", input_path)
    print("Enhanced:", output_path)
    print("Size:", (w, h))

In [ ]:
input_path = "/kaggle/working/vehicle_dataset/Images/Boat/2015_00676.jpg"

output_path = "/kaggle/working/test_00676.jpg"

enhance_one(input_path, output_path)

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

original = Image.open(input_path)
enhanced = Image.open(output_path)

plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
plt.imshow(original)
plt.title("Original ExDark")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(enhanced)
plt.title("RetinexFormer Enhanced")
plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import os
import cv2
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from tqdm import tqdm

# ============================================================
# INPUT / OUTPUT DIRECTORIES
# ============================================================

input_root = "/kaggle/working/vehicle_dataset/Images"

output_root = "/kaggle/working/enhanced_vehicle_dataset/Images"

# Create output directory
os.makedirs(output_root, exist_ok=True)

# Vehicle classes
classes = [
    "Bicycle",
    "Boat",
    "Bus",
    "Car",
    "Motorbike"
]

# ============================================================
# ENHANCEMENT FUNCTION
# ============================================================

def enhance_image(input_path, output_path):

    # Read image
    img = cv2.imread(input_path)

    if img is None:
        return False

    # BGR -> RGB
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Convert to [0, 1]
    img = np.float32(img) / 255.0

    # Convert to tensor
    input_tensor = torch.from_numpy(img).permute(2, 0, 1)
    input_tensor = input_tensor.unsqueeze(0).cuda()

    # Original dimensions
    _, _, h, w = input_tensor.shape

    # RetinexFormer requires divisible by 4
    factor = 4

    new_h = ((h + factor - 1) // factor) * factor
    new_w = ((w + factor - 1) // factor) * factor

    pad_h = new_h - h
    pad_w = new_w - w

    # Padding
    if pad_h > 0 or pad_w > 0:
        input_tensor = F.pad(
            input_tensor,
            (0, pad_w, 0, pad_h),
            mode="reflect"
        )

    # ========================================================
    # RETINEXFORMER
    # ========================================================

    with torch.inference_mode():

        restored = model(input_tensor)

    # Remove padding
    restored = restored[:, :, :h, :w]

    # Clamp
    restored = torch.clamp(restored, 0, 1)

    # Tensor -> NumPy
    restored = (
        restored
        .cpu()
        .permute(0, 2, 3, 1)
        .squeeze(0)
        .numpy()
    )

    # [0,1] -> [0,255]
    restored = (restored * 255).astype(np.uint8)

    # RGB -> BGR for OpenCV
    restored = cv2.cvtColor(restored, cv2.COLOR_RGB2BGR)

    # Save
    cv2.imwrite(output_path, restored)

    return True


# ============================================================
# PROCESS ALL IMAGES
# ============================================================

total = 0
success = 0
failed = []

for class_name in classes:

    input_dir = os.path.join(input_root, class_name)
    output_dir = os.path.join(output_root, class_name)

    os.makedirs(output_dir, exist_ok=True)

    print(f"\nProcessing {class_name}...")

    image_files = []

    for filename in os.listdir(input_dir):

        if filename.lower().endswith(
            (".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG")
        ):
            image_files.append(filename)

    image_files.sort()

    for filename in tqdm(image_files):

        input_path = os.path.join(input_dir, filename)
        output_path = os.path.join(output_dir, filename)

        total += 1

        try:

            result = enhance_image(
                input_path,
                output_path
            )

            if result:
                success += 1
            else:
                failed.append(filename)

        except Exception as e:

            failed.append(
                f"{class_name}/{filename} -> {str(e)}"
            )

# ============================================================
# RESULT
# ============================================================

print("\n" + "=" * 60)
print("RETINEXFORMER ENHANCEMENT COMPLETE")
print("=" * 60)

print("Total images:", total)
print("Successfully enhanced:", success)
print("Failed:", len(failed))

print("\nOutput directory:")
print(output_root)

if failed:
    print("\nFirst 10 failed images:")
    for item in failed[:10]:
        print(item)

In [ ]:
import os

# ============================================================
# PATHS
# ============================================================

original_root = "/kaggle/working/vehicle_dataset/Images"
enhanced_root = "/kaggle/working/enhanced_vehicle_dataset/Images"

missing_log_path = "/kaggle/working/missing_enhanced_images.txt"

valid_extensions = (
    ".jpg",
    ".jpeg",
    ".png",
    ".JPG",
    ".JPEG",
    ".PNG"
)

# ============================================================
# COLLECT ORIGINAL IMAGES
# ============================================================

original_images = []

for class_name in sorted(os.listdir(original_root)):

    class_dir = os.path.join(
        original_root,
        class_name
    )

    if not os.path.isdir(class_dir):
        continue

    for filename in sorted(os.listdir(class_dir)):

        if filename.endswith(valid_extensions):

            relative_path = os.path.join(
                class_name,
                filename
            )

            original_images.append(
                relative_path
            )

# ============================================================
# FIND MISSING ENHANCED IMAGES
# ============================================================

missing_images = []

for relative_path in original_images:

    enhanced_path = os.path.join(
        enhanced_root,
        relative_path
    )

    if not os.path.exists(enhanced_path):

        missing_images.append(
            relative_path
        )

# ============================================================
# SAVE MISSING LIST
# ============================================================

with open(
    missing_log_path,
    "w"
) as f:

    for relative_path in missing_images:
        f.write(relative_path + "\n")

# ============================================================
# SUMMARY
# ============================================================

print("=" * 60)
print("MISSING ENHANCED IMAGE CHECK")
print("=" * 60)

print(
    "Total original images:",
    len(original_images)
)

print(
    "Enhanced images found:",
    len(original_images) - len(missing_images)
)

print(
    "Missing images:",
    len(missing_images)
)

print(
    "\nMissing list saved to:"
)

print(
    missing_log_path
)

print(
    "\nFirst 20 missing images:"
)

for item in missing_images[:20]:
    print(item)

In [ ]:
import os
import cv2
from collections import Counter

original_root = "/kaggle/working/vehicle_dataset/Images"
missing_log_path = "/kaggle/working/missing_enhanced_images.txt"

# ============================================================
# LOAD MISSING LIST
# ============================================================

with open(missing_log_path, "r") as f:
    missing_images = [
        line.strip()
        for line in f
        if line.strip()
    ]

# ============================================================
# INSPECT DIMENSIONS
# ============================================================

dimension_info = []
dimension_counter = Counter()

for relative_path in missing_images:

    img_path = os.path.join(
        original_root,
        relative_path
    )

    img = cv2.imread(img_path)

    if img is None:
        dimension_info.append(
            (relative_path, None, None)
        )
        continue

    h, w = img.shape[:2]

    dimension_info.append(
        (relative_path, h, w)
    )

    dimension_counter[
        (h, w)
    ] += 1

# ============================================================
# SUMMARY
# ============================================================

valid_dims = [
    (h, w)
    for _, h, w in dimension_info
    if h is not None
]

heights = [h for h, w in valid_dims]
widths = [w for h, w in valid_dims]

print("=" * 60)
print("FAILED IMAGE RESOLUTION ANALYSIS")
print("=" * 60)

print("Total missing images:", len(missing_images))
print("Readable images:", len(valid_dims))

if valid_dims:

    print(
        "\nMinimum resolution:",
        f"{min(widths)} x {min(heights)}"
    )

    print(
        "Maximum resolution:",
        f"{max(widths)} x {max(heights)}"
    )

print("\nMost common resolutions:")

for (h, w), count in dimension_counter.most_common(15):

    print(
        f"{w} x {h} -> {count} images"
    )

print("\nFirst 25 missing-image dimensions:")

for relative_path, h, w in dimension_info[:25]:

    if h is None:
        print(relative_path, "-> unreadable")
    else:
        print(
            relative_path,
            "->",
            f"{w} x {h}"
        )

In [ ]:
import os
import gc
import cv2
import numpy as np
import torch
import torch.nn.functional as F

# ============================================================
# PATHS
# ============================================================

original_root = "/kaggle/working/vehicle_dataset/Images"
enhanced_root = "/kaggle/working/enhanced_vehicle_dataset/Images"
missing_log_path = "/kaggle/working/missing_enhanced_images.txt"

retry_failed_log = "/kaggle/working/tiled_retry_failed.txt"

# ============================================================
# TILE SETTINGS
# ============================================================

TILE_SIZE = 512
OVERLAP = 32

device = next(model.parameters()).device

print("=" * 60)
print("TILED RETINEXFORMER RETRY")
print("=" * 60)

print("Device:", device)
print("Tile size:", TILE_SIZE)
print("Overlap:", OVERLAP)

# ============================================================
# LOAD MISSING LIST
# ============================================================

with open(missing_log_path, "r") as f:
    missing_images = [
        line.strip()
        for line in f
        if line.strip()
    ]

print("Images to retry:", len(missing_images))

# ============================================================
# TILE ENHANCEMENT FUNCTION
# ============================================================

def enhance_tile(tile_bgr):

    tile_rgb = cv2.cvtColor(
        tile_bgr,
        cv2.COLOR_BGR2RGB
    )

    tile_rgb = tile_rgb.astype(
        np.float32
    ) / 255.0

    tensor = (
        torch.from_numpy(tile_rgb)
        .permute(2, 0, 1)
        .unsqueeze(0)
        .to(device)
    )

    _, _, h, w = tensor.shape

    # RetinexFormer dimension compatibility
    factor = 4

    new_h = ((h + factor - 1) // factor) * factor
    new_w = ((w + factor - 1) // factor) * factor

    pad_h = new_h - h
    pad_w = new_w - w

    if pad_h > 0 or pad_w > 0:

        tensor = F.pad(
            tensor,
            (0, pad_w, 0, pad_h),
            mode="reflect"
        )

    with torch.inference_mode():

        output = model(tensor)

    output = output[
        :, :, :h, :w
    ]

    output = torch.clamp(
        output,
        0,
        1
    )

    output = (
        output
        .cpu()
        .squeeze(0)
        .permute(1, 2, 0)
        .numpy()
    )

    output = (
        output * 255.0
    ).astype(np.uint8)

    output = cv2.cvtColor(
        output,
        cv2.COLOR_RGB2BGR
    )

    del tensor

    return output


# ============================================================
# FULL IMAGE USING TILES
# ============================================================

def enhance_image_tiled(img):

    h, w = img.shape[:2]

    stride = TILE_SIZE - OVERLAP

    result = np.zeros(
        (h, w, 3),
        dtype=np.float32
    )

    weight = np.zeros(
        (h, w, 1),
        dtype=np.float32
    )

    y_positions = list(
        range(0, max(h - TILE_SIZE, 0) + 1, stride)
    )

    x_positions = list(
        range(0, max(w - TILE_SIZE, 0) + 1, stride)
    )

    if len(y_positions) == 0:
        y_positions = [0]

    if len(x_positions) == 0:
        x_positions = [0]

    if y_positions[-1] + TILE_SIZE < h:
        y_positions.append(
            max(h - TILE_SIZE, 0)
        )

    if x_positions[-1] + TILE_SIZE < w:
        x_positions.append(
            max(w - TILE_SIZE, 0)
        )

    for y in y_positions:

        for x in x_positions:

            y2 = min(
                y + TILE_SIZE,
                h
            )

            x2 = min(
                x + TILE_SIZE,
                w
            )

            tile = img[
                y:y2,
                x:x2
            ]

            enhanced_tile = enhance_tile(
                tile
            )

            th, tw = enhanced_tile.shape[:2]

            result[
                y:y+th,
                x:x+tw
            ] += enhanced_tile.astype(
                np.float32
            )

            weight[
                y:y+th,
                x:x+tw
            ] += 1.0

            del tile
            del enhanced_tile

    result /= np.maximum(
        weight,
        1e-8
    )

    result = np.clip(
        result,
        0,
        255
    ).astype(np.uint8)

    return result


# ============================================================
# RETRY MISSING IMAGES
# ============================================================

success_count = 0
failed_images = []

for idx, relative_path in enumerate(
    missing_images,
    start=1
):

    input_path = os.path.join(
        original_root,
        relative_path
    )

    output_path = os.path.join(
        enhanced_root,
        relative_path
    )

    os.makedirs(
        os.path.dirname(output_path),
        exist_ok=True
    )

    print(
        f"[{idx}/{len(missing_images)}]",
        relative_path
    )

    try:

        img = cv2.imread(
            input_path
        )

        if img is None:
            raise RuntimeError(
                "Image could not be read."
            )

        original_h, original_w = img.shape[:2]

        enhanced = enhance_image_tiled(
            img
        )

        enhanced_h, enhanced_w = enhanced.shape[:2]

        if (
            enhanced_h != original_h
            or enhanced_w != original_w
        ):

            raise RuntimeError(
                "Output dimension mismatch."
            )

        save_ok = cv2.imwrite(
            output_path,
            enhanced
        )

        if not save_ok:

            raise RuntimeError(
                "cv2.imwrite failed."
            )

        success_count += 1

        print(
            "   ✅ Success:",
            f"{original_w}x{original_h}"
        )

    except Exception as e:

        failed_images.append(
            (relative_path, str(e))
        )

        print(
            "   ❌ Failed:",
            str(e)
        )

    finally:

        if "img" in locals():
            del img

        if "enhanced" in locals():
            del enhanced

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


# ============================================================
# SAVE RETRY FAILURE LOG
# ============================================================

with open(
    retry_failed_log,
    "w"
) as f:

    for relative_path, error in failed_images:

        f.write(
            relative_path
            + " -> "
            + error
            + "\n"
        )


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("TILED RETRY COMPLETE")
print("=" * 60)

print(
    "Attempted:",
    len(missing_images)
)

print(
    "Successfully enhanced:",
    success_count
)

print(
    "Still failed:",
    len(failed_images)
)

print(
    "\nFailure log:"
)

print(
    retry_failed_log
)

In [ ]:
import os
import cv2
from collections import Counter

# ============================================================
# PATHS
# ============================================================

original_root = "/kaggle/working/vehicle_dataset/Images"
enhanced_root = "/kaggle/working/enhanced_vehicle_dataset/Images"

valid_extensions = (
    ".jpg", ".jpeg", ".png",
    ".JPG", ".JPEG", ".PNG"
)

# ============================================================
# VALIDATION
# ============================================================

total_original = 0
total_enhanced = 0

missing_files = []
unreadable_original = []
unreadable_enhanced = []
dimension_mismatch = []

class_counts_original = Counter()
class_counts_enhanced = Counter()

# ============================================================
# CHECK EACH ORIGINAL IMAGE
# ============================================================

for class_name in sorted(os.listdir(original_root)):

    original_class_dir = os.path.join(
        original_root,
        class_name
    )

    if not os.path.isdir(original_class_dir):
        continue

    for filename in sorted(os.listdir(original_class_dir)):

        if not filename.endswith(valid_extensions):
            continue

        total_original += 1
        class_counts_original[class_name] += 1

        original_path = os.path.join(
            original_class_dir,
            filename
        )

        enhanced_path = os.path.join(
            enhanced_root,
            class_name,
            filename
        )

        # ----------------------------------------------------
        # CHECK EXISTENCE
        # ----------------------------------------------------

        if not os.path.exists(enhanced_path):

            missing_files.append(
                os.path.join(
                    class_name,
                    filename
                )
            )

            continue

        total_enhanced += 1
        class_counts_enhanced[class_name] += 1

        # ----------------------------------------------------
        # READ IMAGES
        # ----------------------------------------------------

        original_img = cv2.imread(
            original_path
        )

        enhanced_img = cv2.imread(
            enhanced_path
        )

        if original_img is None:

            unreadable_original.append(
                os.path.join(
                    class_name,
                    filename
                )
            )

            continue

        if enhanced_img is None:

            unreadable_enhanced.append(
                os.path.join(
                    class_name,
                    filename
                )
            )

            continue

        # ----------------------------------------------------
        # DIMENSION CHECK
        # ----------------------------------------------------

        original_h, original_w = original_img.shape[:2]
        enhanced_h, enhanced_w = enhanced_img.shape[:2]

        if (
            original_h != enhanced_h
            or original_w != enhanced_w
        ):

            dimension_mismatch.append(
                (
                    os.path.join(
                        class_name,
                        filename
                    ),
                    (original_w, original_h),
                    (enhanced_w, enhanced_h)
                )
            )

# ============================================================
# SUMMARY
# ============================================================

print("=" * 65)
print("FINAL ENHANCED DATASET VALIDATION")
print("=" * 65)

print("\nTotal original images:", total_original)
print("Total enhanced images:", total_enhanced)

print("\nMissing enhanced files:", len(missing_files))
print("Unreadable original files:", len(unreadable_original))
print("Unreadable enhanced files:", len(unreadable_enhanced))
print("Dimension mismatches:", len(dimension_mismatch))

print("\nOriginal class counts:")

for class_name in sorted(class_counts_original):
    print(
        f"{class_name}:",
        class_counts_original[class_name]
    )

print("\nEnhanced class counts:")

for class_name in sorted(class_counts_enhanced):
    print(
        f"{class_name}:",
        class_counts_enhanced[class_name]
    )

# ============================================================
# FINAL STATUS
# ============================================================

print("\n" + "=" * 65)

if (
    total_original == 2997
    and total_enhanced == 2997
    and len(missing_files) == 0
    and len(unreadable_original) == 0
    and len(unreadable_enhanced) == 0
    and len(dimension_mismatch) == 0
):

    print("✅ DATASET VALIDATION PASSED")
    print("All 2997 enhanced images are complete and geometry-safe.")

else:

    print("❌ DATASET VALIDATION FOUND ISSUES")

print("=" * 65)

# ============================================================
# OPTIONAL ERROR DETAILS
# ============================================================

if missing_files:
    print("\nMissing files:")
    for x in missing_files[:20]:
        print(x)

if unreadable_enhanced:
    print("\nUnreadable enhanced files:")
    for x in unreadable_enhanced[:20]:
        print(x)

if dimension_mismatch:
    print("\nFirst dimension mismatches:")
    for item in dimension_mismatch[:20]:
        print(item)

In [ ]:
import os
import shutil

export_root = "/kaggle/working/vehicle_lowlight_final"

if os.path.exists(export_root):
    shutil.rmtree(export_root)

os.makedirs(export_root, exist_ok=True)

shutil.copytree(
    "/kaggle/working/vehicle_dataset",
    os.path.join(export_root, "vehicle_dataset")
)

shutil.copytree(
    "/kaggle/working/enhanced_vehicle_dataset",
    os.path.join(export_root, "enhanced_vehicle_dataset")
)

print("✅ Final export folder created:")
print(export_root)

In [42]:
import os
import shutil

source_folder = "/kaggle/working/vehicle_lowlight_final"
zip_base = "/kaggle/working/vehicle_lowlight_final"

print("Source exists:", os.path.exists(source_folder))

zip_path = shutil.make_archive(
    zip_base,
    "zip",
    root_dir="/kaggle/working",
    base_dir="vehicle_lowlight_final"
)

print("\n✅ ZIP created successfully")
print("ZIP path:", zip_path)

size_gb = os.path.getsize(zip_path) / (1024 ** 3)

print(f"ZIP size: {size_gb:.2f} GB")

Source exists: True

✅ ZIP created successfully
ZIP path: /kaggle/working/vehicle_lowlight_final.zip
ZIP size: 1.51 GB
